# Quickstart: ECMWF AIFS Single forecast, virtual - dynamical.org Icechunk Zarr

ECMWF's Artificial Intelligence Forecasting System (AIFS) is an AI based weather model that produces global forecasts. It generates 15-day forecasts at 0.25 degree resolution every 6 hours.

This is the **virtual** version of the archive: instead of copying the data, it stores byte-range references into ECMWF's original GRIB files and decodes them as you read. That makes it cheap to carry **every** variable ECMWF publishes — all 35, including the six on pressure levels — and optimizes it for spatial (map) access. For time series at a point across many forecasts, prefer the [materialized ECMWF AIFS Single forecast](https://dynamical.org/catalog/ecmwf-aifs-single-forecast/).

Dataset documentation: https://dynamical.org/catalog/ecmwf-aifs-single-forecast-virtual/

Note: `dynamical-catalog>=0.8.0` (or `zarr>=3.2 icechunk>=2.0 gribberish>=1.5`) is required.

Dataset licenced [CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/) and [ECMWF Terms of Use](https://apps.ecmwf.int/datasets/licences/general/).

In [ ]:
# If running locally, follow README.md for simple dependency installation.
# If using Google Colab, run this cell.
!uv pip install dynamical-catalog

In [ ]:
import dynamical_catalog

ds = dynamical_catalog.open("ecmwf-aifs-single-forecast-virtual", chunks=None)
ds

This is a forecast dataset with two time dimensions: `init_time` (when the forecast run began) and `lead_time` (how far ahead the forecast extends). The `valid_time` coordinate combines these to give the actual forecasted time. It also has `latitude` and `longitude` spatial dimensions on a global 0.25 degree grid.

Two things are worth knowing before you start:

- **One chunk is one global field.** Reading any single grid cell decodes the whole map behind it, so maps are cheap and a point series over one forecast is fine (61 fields, a few seconds), but scanning a point across thousands of `init_time`s is expensive.
- **Variables on pressure levels live in a group.** Surface and single-level variables are at the dataset root; the six variables carried on the 14 pressure levels are in the `pressure_level` group.

In [ ]:
# Variables with a vertical dimension live in the pressure_level group
ds_pressure = dynamical_catalog.open(
    "ecmwf-aifs-single-forecast-virtual", group="pressure_level", chunks=None
)
ds_pressure

### Accumulated variables are run totals

Precipitation, snowfall, runoff and the two downward radiation fields accumulate from the start of each forecast run, which is what `run_total` in their names means. So the value at lead time 15 days is the total for the whole forecast, and the amount falling in any window is the difference between its endpoints.

In [ ]:
# 15-day precipitation forecast for Nairobi, Kenya
plot_ds = ds.sel(init_time="2026-03-01T00", latitude=-1.3, longitude=36.8, method="nearest")

accumulated = plot_ds["total_precipitation_run_total_surface"]
_ = accumulated.plot(x="valid_time", figsize=(10, 5))

In [ ]:
# Differencing the run total gives the precipitation falling in each 6 hour step
per_step = accumulated.diff("lead_time")
per_step.attrs["long_name"] = "6-hourly precipitation"
per_step.attrs["units"] = accumulated.attrs["units"]
_ = per_step.plot(x="valid_time", figsize=(10, 5))

### The large scale flow, 5 days ahead

Geopotential height on the 500 hPa surface reveals the ridges and troughs that steer weather systems. It comes from the `pressure_level` group.

In [ ]:
(
    ds_pressure["geopotential_height"]
    .sel(init_time="2025-02-01T00", lead_time="5d", pressure_level=500)
    .sel(latitude=slice(80, 20))
    .plot(figsize=(12, 5), cmap="RdYlBu_r")
)

### How strong was the wind during Storm Éowyn?

Storm Éowyn struck Ireland and the UK on January 24, 2025 with record-breaking winds. We can compute 10m wind speed from the u and v components and map the storm's intensity.

In [ ]:
import numpy as np

# 10m wind speed over the British Isles, AIFS forecast initialized Jan 24 00Z, 6 hours ahead
plot_ds = ds.sel(
    init_time="2025-01-24T00",
    lead_time="6h",
    latitude=slice(70, 40),
    longitude=slice(-26, 15),
)

wind_speed = np.sqrt(plot_ds["wind_u_10m"]**2 + plot_ds["wind_v_10m"]**2)
wind_speed.attrs["units"] = "m s-1"
wind_speed.attrs["long_name"] = "10m wind speed"
wind_speed.plot(figsize=(8, 7), cmap="YlOrRd")

### Where is the coldest temperature forecast in the Northern Hemisphere right now?

Using the most recent forecast initialization, we can find the location with the lowest predicted 2m temperature over the next 24 hours.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

latest_init = ds.init_time[-1].values

cold_forecast = (
    ds["temperature_2m"]
    .sel(init_time=latest_init, lead_time=slice("0h", "23h"))
    .sel(latitude=slice(70, 30))  # Northern Hemisphere mid-latitudes
    .min(dim="lead_time")
    .load()
)

min_val = float(cold_forecast.min())
min_loc = cold_forecast.argmin(dim=["latitude", "longitude"])
min_lat = float(cold_forecast.latitude[min_loc["latitude"]])
min_lon = float(cold_forecast.longitude[min_loc["longitude"]])

fig, ax = plt.subplots(figsize=(10, 2))
cold_forecast.plot(ax=ax, cmap="coolwarm")
# Draw a red circle around the coldest point
ax.plot(min_lon, min_lat, "ro", markersize=12, markerfacecolor="none", markeredgewidth=2)
ax.set_title(f"Coldest 24h forecast: {min_val:.1f} °C at ({min_lat:.1f}°, {min_lon:.1f}°)\nForecast initialized: {pd.Timestamp(latest_init)}")

### Animated map

Total cloud cover traces weather systems as they sweep across the globe. Let's animate a week of forecast steps.

In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

# Animation of total cloud cover over a week-long forecast
data = (
    ds["total_cloud_cover_atmosphere"]
    .sel(
        init_time="2026-03-01T00",
        lead_time=slice("6h", "7d"),
        latitude=slice(70, -70),
    )
    .load()
)

scale = 0.4
dpi = 80
fig, ax = plt.subplots(figsize=(data.longitude.size * scale / dpi, data.latitude.size * scale / dpi), dpi=dpi)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.axis("off")

img = ax.imshow(data.isel(lead_time=0), cmap="Blues_r", vmin=0, vmax=100)
anim = FuncAnimation(fig=fig, frames=data, func=lambda frame: img.set_data(frame), interval=200)

HTML(anim.to_jshtml())

### Community challenge

This archive carries every variable ECMWF publishes for AIFS Single, including fields the materialized dataset leaves out — soil temperature and moisture on two layers, the four cloud-cover fields, snow, runoff, and the full pressure-level suite.

Here are a few for inspiration:
- Build a vertical cross-section. Pull `temperature` and `specific_humidity` from the `pressure_level` group along a line of longitude and plot them against pressure to reveal fronts and the tropopause.
- Track a tropical cyclone by extracting minimum sea level pressure or maximum wind speed in a bounding box at each lead time (e.g. Typhoon Yagi, September 2024). How does the predicted storm track shift across initializations?
- Compare soil moisture in the two soil layers through a forecast of a heavy rain event. How far down does the wetting signal reach, and how quickly?

For point time series across many forecast initializations, the [materialized ECMWF AIFS Single forecast](https://dynamical.org/catalog/ecmwf-aifs-single-forecast/) is chunked for that access pattern.